# 02 - 记忆系统深度解析

本notebook深入介绍Agent记忆管理的不同策略和实现原理。

## 学习目标
- 理解记忆系统的理论基础
- 掌握不同记忆类型的实现细节
- 学会选择合适的记忆策略
- 理解向量检索和语义记忆
- 掌握记忆压缩和优化技术

## 1. 理论背景

### 记忆的层次结构

在认知科学中，记忆分为多个层次：

```
┌─────────────────────────────────────────────────┐
│              感知记忆 (Sensory Memory)          │
│           ← 瞬时，0.5-3秒                       │
├─────────────────────────────────────────────────┤
│              工作记忆 (Working Memory)          │
│           ← 短期，15-30秒                       │
├─────────────────────────────────────────────────┤
│              长期记忆 (Long-term Memory)        │
│    ┌────────────┬─────────────────────────────┐ │
│    │ 陈述性记忆 │           程序性记忆         │ │
│    │  (事实)   │           (技能)            │ │
│    └────────────┴─────────────────────────────┘ │
└─────────────────────────────────────────────────┘
```

### LLM中的记忆类型

1. **短期记忆**: 对话上下文窗口内
2. **长期记忆**: 向量数据库中的知识存储
3. **混合记忆**: 结合摘要和检索

In [ ]:
import sys
sys.path.insert(0, '../src')

from memory import (
    Message, MessageRole,
    BufferMemory, WindowMemory,
    SummaryMemory, VectorMemory
)
import json
from typing import List, Dict, Any
from datetime import datetime

## 2. Message 消息类详解

### Message 的基本结构

In [ ]:
# 创建不同类型的消息
system_msg = Message.system("你是一个Python专家助手")
user_msg = Message.user("如何使用列表推导式？")
assistant_msg = Message.assistant("列表推导式是Python中创建列表的简洁语法...")

# 查看消息属性
print("=== 系统消息 ===")
print(f"角色: {system_msg.role.value}")
print(f"内容: {system_msg.content}")
print(f"时间戳: {system_msg.timestamp}")
print(f"Token数: {system_msg.token_count}")
print(f"元数据: {system_msg.metadata}")

print("\n=== 转换为字典 ===")
print(json.dumps(system_msg.to_dict(), indent=2, ensure_ascii=False))

In [ ]:
# 创建带元数据的消息
msg_with_metadata = Message(
    role=MessageRole.USER,
    content="帮我分析这段代码",
    metadata={
        "language": "python",
        "topic": "code_review",
        "priority": "high"
    }
)
print(f"消息: {msg_with_metadata.content}")
print(f"元数据: {msg_with_metadata.metadata}")

In [ ]:
# 从字典创建消息
msg_dict = {
    "role": "user",
    "content": "今天天气怎么样？",
    "timestamp": "2024-01-01T10:00:00"
}
restored_msg = Message.from_dict(msg_dict)
print(f"恢复的消息: {restored_msg.role.value} - {restored_msg.content}")

## 3. BufferMemory 缓冲记忆

保存所有对话历史，是最简单直接的记忆方式。

In [ ]:
# 创建缓冲记忆
buffer = BufferMemory(system_message="你是一个AI编程助手")

print("=== 初始状态 ===")
print(f"消息数量: {buffer.message_count}")
print(f"估算Token: {buffer.token_count}")
print(f"系统消息: {buffer.system_message}")

In [ ]:
# 添加对话
conversations = [
    ("user", "什么是装饰器？"),
    ("assistant", "装饰器是Python中用于修改函数行为的工具..."),
    ("user", "能给个例子吗？"),
    ("assistant", "当然！这是一个简单的装饰器示例：\n```python\ndef my_decorator(func):\n    def wrapper():\n        print('Before')\n        func()\n        print('After')\n    return wrapper```"),
    ("user", "装饰器可以带参数吗？"),
    ("assistant", "可以！带参数的装饰器需要多一层嵌套..."),
]

for role, content in conversations:
    if role == "user":
        buffer.add_user_message(content)
    else:
        buffer.add_assistant_message(content)

print(f"添加对话后消息数: {buffer.message_count}")
print(f"估算Token: {buffer.token_count}")

In [ ]:
# 获取消息
messages = buffer.get_messages()
print("=== 所有消息 ===")
for i, msg in enumerate(messages, 1):
    role_emoji = "👤" if msg.role == MessageRole.USER else "🤖" if msg.role == MessageRole.ASSISTANT else "⚙️"
    preview = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
    print(f"{i}. {role_emoji} [{msg.role.value}]: {preview}")

In [ ]:
# 只获取用户消息
user_messages = buffer.get_user_messages()
print(f"\n用户消息数: {len(user_messages)}")
for msg in user_messages:
    print(f"  - {msg.content}")

In [ ]:
# 只获取助手消息
assistant_messages = buffer.get_assistant_messages()
print(f"\n助手消息数: {len(assistant_messages)}")
for msg in assistant_messages:
    preview = msg.content[:50] + "..." if len(msg.content) > 50 else msg.content
    print(f"  - {preview}")

In [ ]:
# 获取最近的k条消息
recent = buffer.get_recent_messages(k=3)
print(f"\n最近3条消息:")
for msg in recent:
    role_emoji = "👤" if msg.role == MessageRole.USER else "🤖"
    print(f"  {role_emoji} {msg.content[:50]}...")

In [ ]:
# 清空消息（保留系统消息）
buffer.clear()
print(f"\n清空后:")
print(f"消息数量: {buffer.message_count}")
print(f"系统消息保留: {buffer.system_message}")

## 4. WindowMemory 窗口记忆

滑动窗口策略只保留最近的对话，控制上下文长度。

In [ ]:
# 创建窗口记忆
window = WindowMemory(k=3, system_message="你是AI助手")

print("=== 窗口记忆演示 ===")
print(f"窗口大小k: {window.window_size}")
print(f"最大保留消息数: {window.max_messages}")

In [ ]:
# 添加多条消息观察窗口行为
questions = [
    "Python是什么？",
    "列表和元组的区别？",
    "如何读写文件？",
    "什么是生成器？",
    "异常处理怎么写？",
    "如何使用类和对象？"
]

for i, q in enumerate(questions, 1):
    window.add_user_message(q)
    window.add_assistant_message(f"这是对问题{i}的回答...")
    
    current_count = len([m for m in window.get_messages() if m.role != MessageRole.SYSTEM])
    print(f"添加第{i}轮后: 保留{current_count}条消息")
    
    if i % 2 == 0:
        print(f"  当前窗口内容:")
        for msg in window.get_messages():
            if msg.role != MessageRole.SYSTEM:
                print(f"    [{msg.role.value}]: {msg.content[:30]}...")

In [ ]:
# 查看最终保留的消息
print("\n=== 最终保留的消息 ===")
final_messages = [m for m in window.get_messages() if m.role != MessageRole.SYSTEM]
for msg in final_messages:
    print(f"[{msg.role.value}]: {msg.content}")

In [ ]:
# 不同k值的对比
print("\n=== 不同k值对比 ===")
for k in [1, 2, 5, 10]:
    w = WindowMemory(k=k, system_message="AI助手")
    for i in range(10):
        w.add_user_message(f"消息{i}")
    msg_count = len([m for m in w.get_messages() if m.role != MessageRole.SYSTEM])
    print(f"k={k:2d} -> 保留{msg_count}条消息")

## 5. SummaryMemory 摘要记忆

当对话过长时自动压缩历史为摘要，保留关键信息。

In [ ]:
# 创建摘要记忆
summary = SummaryMemory(max_messages=6)

print("=== 摘要记忆演示 ===")
print(f"最大消息数: {summary.max_messages}")
print(f"触发摘要阈值: {summary.max_messages - 2}")

In [ ]:
# 添加消息观察摘要生成
long_conversation = [
    "用户询问了Python的基本语法",
    "解释了Python的变量和数据类型",
    "用户询问了控制流程",
    "解释了if/else和循环语句",
    "用户询问了函数定义",
    "解释了def关键字和参数",
    "用户询问了面向对象",
    "解释了类和继承的概念",
]

for i, msg_text in enumerate(long_conversation, 1):
    if i % 2 == 1:
        summary.add_user_message(msg_text)
    else:
        summary.add_assistant_message(msg_text)
    
    current_summary = summary.summary
    has_summary = current_summary is not None
    print(f"添加第{i}条: 有摘要={has_summary}")
    
    if has_summary:
        print(f"  当前摘要: {current_summary[:80]}...")

In [ ]:
# 查看最终状态
print(f"\n=== 最终摘要 ===")
print(f"摘要内容: {summary.summary}")

print(f"\n=== 当前消息 ===")
for msg in summary.get_messages():
    if msg.role != MessageRole.SYSTEM:
        print(f"[{msg.role.value}]: {msg.content[:50]}...")

In [ ]:
# 获取完整上下文（摘要+当前消息）
print("\n=== 完整上下文 ===")
full_context = summary.get_context()
print(full_context)

## 6. VectorMemory 向量记忆

使用向量相似度进行语义检索，找到与查询相关的历史消息。

In [ ]:
# 创建向量记忆
vector = VectorMemory(top_k=3)

print("=== 向量记忆演示 ===")
print(f"返回最相关: {vector.top_k}条消息")

In [ ]:
# 添加不同主题的消息
knowledge_base = [
    "Python的列表是可变的，用方括号[]创建",
    "Python的元组是不可变的，用圆括号()创建",
    "pandas是Python的数据分析库，提供DataFrame数据结构",
    "numpy是Python的数值计算库，支持多维数组",
    "机器学习中的过拟合是指模型在训练集上表现好但泛化能力差",
    "深度学习使用神经网络，特别适合处理图像和文本数据",
    "装饰器是Python中修改函数行为的语法糖",
    "生成器是使用yield关键字的惰性迭代器",
    "Git是分布式版本控制系统",
    "Docker是容器化平台，可以打包应用和依赖",
]

for text in knowledge_base:
    vector.add_user_message(text)

print(f"已添加{len(knowledge_base)}条知识")

In [ ]:
# 测试语义检索
queries = [
    "Python数据结构",
    "数据科学库",
    "神经网络",
    "开发工具",
]

for query in queries:
    relevant = vector.retrieve(query)
    print(f"\n查询: {query}")
    print("相关消息:")
    for i, msg in enumerate(relevant, 1):
        print(f"  {i}. {msg.content}")

In [ ]:
# 获取所有消息和检索统计
print(f"\n总消息数: {len(vector.get_messages())}")
print(f"检索的top_k: {vector.top_k}")

## 7. 记忆类型对比和选择

In [ ]:
# 创建对比实验
conversations = [f"这是第{i}条消息，内容是关于主题{i%3}" for i in range(20)]

# BufferMemory
buffer = BufferMemory()
for msg in conversations:
    buffer.add_user_message(msg)

# WindowMemory
window = WindowMemory(k=5)
for msg in conversations:
    window.add_user_message(msg)

# SummaryMemory
summary = SummaryMemory(max_messages=8)
for msg in conversations:
    summary.add_user_message(msg)

print("=== 记忆类型对比 ===")
print(f"{'类型':<15} {'消息数':<10} {'Token数':<10} {'保留内容'}")
print("-" * 60)
print(f"{'BufferMemory':<15} {buffer.message_count:<10} {buffer.token_count:<10} 所有历史")
print(f"{'WindowMemory':<15} {len([m for m in window.get_messages() if m.role != MessageRole.SYSTEM]):<10} {window.token_count:<10} 最近5轮")
print(f"{'SummaryMemory':<15} {len([m for m in summary.get_messages() if m.role != MessageRole.SYSTEM]):<10} {summary.token_count:<10} 摘要+最新")

## 8. 记忆策略决策树

In [ ]:
def select_memory_strategy(
    conversation_length: int,
    token_limit: int,
    need_semantic_search: bool = False,
    preserve_early_info: bool = False
) -> str:
    """
    根据场景选择合适的记忆策略
    
    Args:
        conversation_length: 对话轮数
        token_limit: Token限制
        need_semantic_search: 是否需要语义检索
        preserve_early_info: 是否需要保留早期信息
    
    Returns:
        推荐的记忆类型
    """
    if need_semantic_search:
        return "VectorMemory - 需要语义检索时使用"
    
    if conversation_length < 5:
        return "BufferMemory - 短对话，保留完整历史"
    
    if conversation_length < 20:
        if token_limit < 4000:
            return "WindowMemory(k=10) - 中等对话，控制长度"
        return "BufferMemory - Token充足，保留完整历史"
    
    if preserve_early_info:
        return "SummaryMemory - 长对话，需要保留早期信息"
    
    return "WindowMemory(k=5) - 长对话，只保留最近内容"

# 测试决策函数
scenarios = [
    (3, 8000, False, False),
    (15, 4000, False, False),
    (30, 8000, False, True),
    (50, 2000, False, False),
    (20, 8000, True, False),
]

print("=== 记忆策略推荐 ===")
for length, tokens, semantic, preserve in scenarios:
    strategy = select_memory_strategy(length, tokens, semantic, preserve)
    print(f"\n场景: 轮数={length}, Token={tokens}, 语义检索={semantic}, 保留早期={preserve}")
    print(f"推荐: {strategy}")

## 9. 高级：混合记忆系统

结合多种记忆类型的优势，构建更强大的记忆系统。

In [ ]:
class HybridMemory:
    """
    混合记忆系统：结合短期和长期记忆
    - 短期：WindowMemory保存最近对话
    - 长期：VectorMemory存储重要信息
    """
    
    def __init__(self, window_size: int = 5, vector_top_k: int = 3):
        self.short_term = WindowMemory(k=window_size)
        self.long_term = VectorMemory(top_k=vector_top_k)
        self._important_keywords = ["重要", "记住", "关键", "必须"]
    
    def add_message(self, role: str, content: str):
        # 添加到短期记忆
        if role == "user":
            self.short_term.add_user_message(content)
        else:
            self.short_term.add_assistant_message(content)
        
        # 检查是否需要添加到长期记忆
        if self._is_important(content):
            if role == "user":
                self.long_term.add_user_message(content)
            else:
                self.long_term.add_assistant_message(content)
    
    def _is_important(self, content: str) -> bool:
        # 简单的关键词判断
        return any(kw in content for kw in self._important_keywords)
    
    def get_context(self, query: str = None) -> str:

In [ ]:
        # 获取短期记忆
        context_parts = ["【最近对话】"]
        for msg in self.short_term.get_messages():
            if msg.role != MessageRole.SYSTEM:
                context_parts.append(f"{msg.role.value}: {msg.content}")
        
        # 如果有查询，添加相关长期记忆
        if query:
            relevant = self.long_term.retrieve(query)
            if relevant:
                context_parts.append("\n【相关记忆】")
                for msg in relevant:
                    context_parts.append(f"- {msg.content}")
        
        return "\n".join(context_parts)

# 使用混合记忆
hybrid = HybridMemory(window_size=3, vector_top_k=2)

# 模拟对话
hybrid.add_message("user", "我的名字是张三")
hybrid.add_message("assistant", "你好张三！")
hybrid.add_message("user", "记住，我必须使用Python 3.9版本")
hybrid.add_message("assistant", "好的，我会记住你使用Python 3.9")
hybrid.add_message("user", "推荐一个数据分析库")
hybrid.add_message("assistant", "推荐使用pandas")

# 获取上下文
context = hybrid.get_context(query="Python版本")
print(context)

## 10. 记忆持久化

In [ ]:
import json
from typing import List

class PersistentMemory:
    """支持持久化的记忆系统"""
    
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.buffer = BufferMemory()
        self._load()
    
    def _load(self):
        """从文件加载记忆"""
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for msg_data in data.get('messages', []):
                    msg = Message.from_dict(msg_data)
                    self.buffer._messages.append(msg)
            print(f"加载了{len(self.buffer._messages)}条历史消息")
        except FileNotFoundError:
            print("未找到历史记录，开始新会话")
    
    def save(self):
        """保存记忆到文件"""
        data = {
            'messages': [msg.to_dict() for msg in self.buffer.get_messages()],
            'saved_at': datetime.now().isoformat()
        }
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"保存了{len(data['messages'])}条消息")
    
    def add_user_message(self, content: str):
        self.buffer.add_user_message(content)
        self.save()
    
    def add_assistant_message(self, content: str):
        self.buffer.add_assistant_message(content)
        self.save()
    
    def get_messages(self) -> List[Message]:
        return self.buffer.get_messages()

# 使用持久化记忆（注意：实际保存需要有效路径）
# persistent = PersistentMemory('/tmp/chat_history.json')
# persistent.add_user_message("测试消息")
# persistent.add_assistant_message("这是回复")

print("持久化记忆类已定义（示例中未实际保存）")

## 11. Token使用分析

In [ ]:
def analyze_token_usage(memory_class, messages: List[str], **kwargs):
    """分析不同记忆类型的Token使用"""
    memory = memory_class(**kwargs)
    
    for msg in messages:
        memory.add_user_message(msg)
    
    total_messages = len([m for m in memory.get_messages() if m.role != MessageRole.SYSTEM])
    total_tokens = memory.token_count
    avg_tokens = total_tokens / total_messages if total_messages > 0 else 0
    
    return {
        "type": memory_class.__name__,
        "messages": total_messages,
        "total_tokens": total_tokens,
        "avg_tokens_per_msg": avg_tokens
    }

# 测试数据
test_messages = [f"这是第{i}条测试消息，包含一些中文和English内容" for i in range(20)]

# 对比分析
results = [
    analyze_token_usage(BufferMemory, test_messages),
    analyze_token_usage(WindowMemory, test_messages, k=5),
    analyze_token_usage(WindowMemory, test_messages, k=10),
    analyze_token_usage(SummaryMemory, test_messages, max_messages=10),
]

print("=== Token使用分析 ===")
print(f"{'类型':<20} {'消息数':<10} {'总Token':<12} {'平均Token'}")
print("-" * 60)
for r in results:
    print(f"{r['type']:<20} {r['messages']:<10} {r['total_tokens']:<12} {r['avg_tokens_per_msg']:.1f}")

## 总结

### 记忆类型对比表

| 记忆类型 | 优点 | 缺点 | 适用场景 |
|---------|------|------|----------|
| BufferMemory | 完整历史，简单直接 | Token消耗大 | 短对话（<10轮） |
| WindowMemory | 控制长度，性能好 | 丢失早期信息 | 中等对话（10-50轮） |
| SummaryMemory | 压缩历史，保留关键 | 信息可能有损失 | 长对话（>50轮） |
| VectorMemory | 语义检索，知识密集 | 需要嵌入模型 | 知识检索场景 |

### 选择建议

```
对话长度 → Token限制 → 推荐策略
─────────────────────────────────────
< 10轮    → 无限制      → BufferMemory
10-30轮   → < 4000      → WindowMemory(k=10)
30-50轮   → < 8000      → WindowMemory(k=5)
> 50轮    → 任意        → SummaryMemory
知识检索  → 任意        → VectorMemory
混合需求  → 任意        → HybridMemory
```

### 最佳实践

1. **定期检查Token使用**，避免超出上下文窗口
2. **系统消息始终保留**，确保Agent行为一致
3. **重要信息单独存储**，如用户偏好、关键事实
4. **考虑异步持久化**，避免阻塞对话流程
5. **使用混合策略**，结合短期和长期记忆优势